In [2]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, make_scorer, f1_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
import numpy as np

In [3]:
df = pd.read_csv('./churd.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df = df.drop(['customerID'], axis=1)

In [5]:
df

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.5,No
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.9,No
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.6,Yes


In [6]:
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'], errors='coerce'
).astype(np.float64)

In [7]:
df.isna().sum()

gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [8]:
df = df.dropna()

In [9]:
print(df.isnull().sum())

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [10]:
X = df.drop(columns=['Churn'])
y = df['Churn']

In [11]:
categorical_columns = X.select_dtypes(include=['object', 'str', 'category']).columns.tolist()

numerical_columns = X.select_dtypes(include=['number']).columns.tolist()

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'encoder',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_columns
        ),
        (
            'numeric',
            StandardScaler(),#'passthrough',
            numerical_columns
        )
    ],
    remainder='drop'
)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
param_grid = {
    'svm': {
        # 'model': svm.SVC(gamma='auto', probability=True),
        'params': {
            'svc__C': [0.1, 1, 10, 100],
            'svc__kernel': ['linear', 'rbf'],
            'svc__class_weight': [None, 'balanced']
        }
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params': {
            'randomforestclassifier__n_estimators': [100, 200, 300],
            'randomforestclassifier__max_depth': [None, 5, 10, 20],
            'randomforestclassifier__min_samples_split': [2, 5, 10],
            'randomforestclassifier__min_samples_leaf': [6, 15, 30],
        },
    },
    'logistic_regression': {
        'model': LogisticRegression(solver='lbfgs'),
        'params': {
            'logisticregression__C': [0.01, 0.1, 1, 10],
        }
    }
}

params_grid = {
    'logisticregression__C': [0.01, 0.1, 1, 10]
}

In [29]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score


def train_model(Xt, yt, Xte, yte):

    # scores = []
    # best_estimator = {}

    # # for loop

    # for algorithm, mp in param_grid.items():
        #  pipeline

    pipeline = make_pipeline(
        preprocessor,
        LogisticRegression()
    )

    # print(pipeline.named_steps.keys())

    # gridsearchcv but iterate throughh it

    f1_scorer = make_scorer(
        f1_score,
        pos_label='Yes'
    )

    clf = GridSearchCV(
        estimator=pipeline,
        param_grid=params_grid,
        cv=5,
        n_jobs=-1,
        verbose=1,
        scoring='accuracy',
        error_score='raise'
    )
    clf.fit(Xt, yt)
    # scores.append({
    #     'model': algorithm,
    #     'best_score': clf.best_score_,
    #     'best_params': clf.best_params_
    # })

    # best_estimator[algorithm] = clf.best_estimator_

    best_model = clf.best_estimator_

    prediction = clf.predict(Xte)
    

    print('Accuracy:', accuracy_score(yte, prediction))
    print('F1 Score:', f1_score(yte, prediction, pos_label='Yes'))
    print('\nClassification Report:\n',
        classification_report(yte, prediction))

    print('\nConfusion Matrix:\n',
      confusion_matrix(yte, prediction, labels=['No', 'Yes']))
    

    # y_pred = best_model.predict(Xte)

    # print("Selected model:", model_name)
    # print("Accuracy:", accuracy_score(yte, y_pred))
    # print("F1 score:", f1_score(yte, y_pred, pos_label='Yes'))
    # print("\nClassification report:")
    # print(classification_report(yte, y_pred))
    # print("\nConfusion matrix:")
    # print(confusion_matrix(yte, y_pred))

    # results = pd.DataFrame(scores)
    # print("\nGrid search results:")
    # print(results.to_string(index=False))

        # fitting and get mae and mse

    return best_model



In [30]:
pred = train_model(X_train, y_train, X_test, y_test)

# ('preprocessor', preprocessor),

Fitting 5 folds for each of 4 candidates, totalling 20 fits


Accuracy: 0.7882018479033405
F1 Score: 0.564327485380117

Classification Report:
               precision    recall  f1-score   support

          No       0.84      0.89      0.86      1033
         Yes       0.62      0.52      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407


Confusion Matrix:
 [[916 117]
 [181 193]]


In [25]:
import joblib
# from pathlib import Path

In [31]:
# Path('artifacts').mkdir(exist_ok=True)

joblib.dump(
    pred,
    'artifacts/logisticregression_churn.pkl'
)

print('LOGISTIC REGRESSION Model Saved')

LOGISTIC REGRESSION Model Saved
